# 05 · Near-duplicate predictors, and the clean pool

Several of the 95 predictors carry the same information twice: wealth as a quintile and as the
continuous score's quintile, education in four levels and in single years, region as a code
and as a label, travel time in minutes and as its log. For explanation this matters: a model
that splits wealth between two copies shows each copy as half as important as wealth is.

Every pair of predictors is scored on the development women (Spearman correlation for numbers,
Theil's U for categories). A pair above 0.95 is a **near-duplicate**, and one of the two is
dropped: 10 of the 95. Two more pairs are exact one-to-one recodings between the 21's own
constructs and pool columns (household size, travel time to a hospital); the pool keeps the
21's member of each. The result is the **clean pool of 86 predictors**, the largest of the four
predictor sets.

**Writes:** `outputs/enhanced/redundancy/`

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

In [ ]:
repro.run_script("run_redundancy_analysis.py")

In [ ]:
study = repro.reference("predictor_sets.json")
summary = json.load(open(P.outputs / "enhanced" / "redundancy" / "redundancy_summary.json"))
repro.check("the same 10 near-duplicates are dropped", sorted(summary["dropped"]) == sorted(study["duplicates_dropped"]),
            ", ".join(summary["dropped"]))

# the two exact recodings the 21's constructs make of pool columns
frame = lib.raw_frame()
for a, b in lib.CONSTRUCT_TWINS.items():
    s = frame[[a, b]]
    one_to_one = s.groupby(a, dropna=False)[b].nunique(dropna=False).max() == 1 and s.groupby(b, dropna=False)[a].nunique(dropna=False).max() == 1
    repro.check(f"{a} is an exact recoding of {b}", one_to_one)

num, cat = lib.clean_pool()
repro.check("the clean pool has 86 predictors", len(num) + len(cat) == 86, f"{len(num)} numeric, {len(cat)} categorical")
repro.report()